# SQL Query Optimization — Complete Reference

| Pattern | Technique |
|---------|----------|
| EXPLAIN plan | Read query plan, identify full scans |
| Index selection | Composite index column order, covering index |
| Join strategies | Nested loop vs hash vs merge join |
| Predicate pushdown | Filter early, reduce data before joins |
| Query rewriting | Correlated subquery → JOIN, OR → UNION |

**Mental model**: The query optimizer converts SQL into a physical execution plan. Optimization = reducing rows processed early (filters, indexes) and choosing efficient join algorithms.

```
Optimizer pipeline:
  SQL text → parse → logical plan → cost-based optimization → physical plan → execute
  
Key levers:
  1. Indexes          — skip rows without scanning
  2. Statistics       — optimizer uses row counts, cardinality
  3. Predicate order  — filter before join (pushdown)
  4. Join algorithm   — nested loop (small), hash (large), merge (pre-sorted)
```

## Visual Model

```
QUERY PLAN READING (EXPLAIN output)
────────────────────────────────────
  SCAN table orders         ← full table scan — BAD if table is large
  SEARCH orders USING INDEX idx_customer (customer_id=?)  ← index seek — GOOD
  SEARCH orders USING COVERING INDEX  ← all cols in index, no table lookup — BEST

INDEX ANATOMY
──────────────
  Single column:    CREATE INDEX idx_a ON t(a)
  Composite:        CREATE INDEX idx_ab ON t(a, b)
  Covering:         CREATE INDEX idx_cover ON t(a, b, c)  -- c = selected column

  Composite column order rule:
    equality predicates first → range predicates last
    WHERE a = 1 AND b > 5 → index(a, b)  ✓
    WHERE b > 5 AND a = 1 → index(a, b)  ✓  (optimizer reorders)
    WHERE b > 5            → index(b, a)  — only b usable, a skipped

JOIN ALGORITHM SELECTION
─────────────────────────
  Nested Loop:  for each outer row, scan inner
                O(N × M)  — good when inner is small or indexed

  Hash Join:    hash build side into memory, probe with other side
                O(N + M)  — good for large unsorted tables

  Merge Join:   scan two sorted inputs simultaneously
                O(N + M) + sort cost — good when both sides already sorted/indexed
```

## Setup — Libraries and Config

In [ ]:
import sqlite3
import random
import time

print(f"sqlite3 version: {sqlite3.sqlite_version}")

def make_db(n_orders=50000, n_customers=1000, n_products=200):
    """Create database with enough rows to make index differences visible."""
    conn = sqlite3.connect(":memory:")
    conn.row_factory = sqlite3.Row
    cur = conn.cursor()

    # customers
    cur.execute("""
        CREATE TABLE customers(
            customer_id INTEGER PRIMARY KEY,
            name TEXT, region TEXT, tier TEXT
        )
    """)
    regions = ['East','West','Central','North','South']
    tiers = ['Gold','Silver','Bronze']
    cur.executemany("INSERT INTO customers VALUES(?,?,?,?)",
        [(i, f"Cust_{i}", random.choice(regions), random.choice(tiers))
         for i in range(1, n_customers+1)])

    # products
    cur.execute("""
        CREATE TABLE products(
            product_id INTEGER PRIMARY KEY,
            name TEXT, category TEXT, price REAL
        )
    """)
    cats = ['Electronics','Clothing','Food','Tools']
    cur.executemany("INSERT INTO products VALUES(?,?,?,?)",
        [(i, f"Prod_{i}", random.choice(cats), round(random.uniform(5, 500), 2))
         for i in range(1, n_products+1)])

    # orders
    cur.execute("""
        CREATE TABLE orders(
            order_id INTEGER PRIMARY KEY,
            customer_id INTEGER,
            product_id INTEGER,
            order_date TEXT,
            amount REAL,
            status TEXT
        )
    """)
    statuses = ['completed','cancelled','pending']
    import datetime
    base = datetime.date(2023, 1, 1)
    orders = [
        (i,
         random.randint(1, n_customers),
         random.randint(1, n_products),
         str(base + datetime.timedelta(days=random.randint(0, 364))),
         round(random.uniform(10, 2000), 2),
         random.choice(statuses))
        for i in range(1, n_orders+1)
    ]
    cur.executemany("INSERT INTO orders VALUES(?,?,?,?,?,?)", orders)
    conn.commit()
    return conn

def explain(conn, sql, title=""):
    rows = conn.execute(f"EXPLAIN QUERY PLAN {sql}").fetchall()
    if title:
        print(f"\n=== EXPLAIN: {title} ===")
    for r in rows:
        print(f"  [{r[0]},{r[1]},{r[2]}] {r[3]}")

def time_query(conn, sql, label, n=3):
    times = []
    for _ in range(n):
        t0 = time.perf_counter()
        conn.execute(sql).fetchall()
        times.append((time.perf_counter() - t0) * 1000)
    avg = sum(times) / len(times)
    print(f"  {label}: {avg:.2f}ms avg")

conn = make_db()
print("Database ready with 50k orders.")

## Decision Map — Optimization Checklist

```
Query is slow — where to look?
│
├─ Run EXPLAIN QUERY PLAN first
│   ├─ SCAN (no index)?      → Add WHERE-column index
│   ├─ Many rows before JOIN? → Push filter down (CTE/subquery with WHERE)
│   └─ TEMP B-TREE (sort)?   → Add index matching ORDER BY
│
├─ Index exists but not used?
│   ├─ Function on column?   → WHERE DATE(col) = ?  defeats index
│   │                          Use WHERE col BETWEEN ? AND ? instead
│   ├─ Leading column wrong? → Composite index: equality cols first
│   └─ Low selectivity?      → Index on boolean ignored; filter another way
│
├─ JOIN is slow?
│   ├─ Inner table large?    → Index join column on larger table
│   ├─ Joining on expression? → Materialize expression first (CTE)
│   └─ OR condition?         → Rewrite as UNION ALL
│
└─ Aggregation slow?
    ├─ COUNT(*) on big table? → COUNT(*) is fast; avoid COUNT(DISTINCT col)
    ├─ GROUP BY before JOIN?  → Push GROUP BY into CTE first
    └─ HAVING vs WHERE?       → WHERE filters before aggregate (faster)
```

## Pattern 1 — Reading EXPLAIN QUERY PLAN

In [ ]:
# EXPLAIN QUERY PLAN shows how SQLite will execute the query
# Key terms: SCAN = full table scan, SEARCH = index lookup

# Full scan — no index on customer_id yet
explain(conn,
    "SELECT * FROM orders WHERE customer_id = 42",
    "Filter without index")

# Add index
conn.execute("CREATE INDEX idx_orders_customer ON orders(customer_id)")

# Index seek — SEARCH with index
explain(conn,
    "SELECT * FROM orders WHERE customer_id = 42",
    "Filter WITH index")

# Covering index — all needed columns in index, no table lookup
conn.execute("CREATE INDEX idx_orders_cover ON orders(customer_id, order_date, amount)")
explain(conn,
    "SELECT order_date, amount FROM orders WHERE customer_id = 42 ORDER BY order_date",
    "Covering index (no table lookup)")

# Benchmark: scan vs index
print("\nBenchmark (50k rows):")
conn_no_idx = make_db()
time_query(conn_no_idx, "SELECT COUNT(*) FROM orders WHERE customer_id = 42", "No index")
conn_no_idx.execute("CREATE INDEX i ON orders(customer_id)")
time_query(conn_no_idx, "SELECT COUNT(*) FROM orders WHERE customer_id = 42", "With index")

## Pattern 2 — Index Selection and Composite Indexes

In [ ]:
# Composite index column order: equality predicates BEFORE range predicates
# Rule: leftmost prefix must be used; columns after a range are not seekable

conn2 = make_db()

# Query pattern: customer_id = ? AND amount > ? AND status = ?
# Best index for this: (customer_id, status, amount) — two equalities first, range last

# Wrong order: range column (amount) in middle blocks status
conn2.execute("CREATE INDEX idx_wrong ON orders(customer_id, amount, status)")
explain(conn2,
    "SELECT * FROM orders WHERE customer_id=42 AND amount>500 AND status='completed'",
    "Wrong index order (amount before status)")

conn2.execute("DROP INDEX idx_wrong")

# Correct order: equality cols (customer_id, status) first, range (amount) last
conn2.execute("CREATE INDEX idx_correct ON orders(customer_id, status, amount)")
explain(conn2,
    "SELECT * FROM orders WHERE customer_id=42 AND status='completed' AND amount>500",
    "Correct index order (equalities first, range last)")

print("""
Index prefix rule:
  index(a, b, c) usable when WHERE includes:
    a = ?              → uses a only
    a = ? AND b = ?    → uses a, b
    a = ? AND b > ?    → uses a for equality, b for range (c skipped)
    b = ?              → NOT usable (a is missing from prefix)
    a = ? AND c = ?    → uses a only (b gap breaks the chain)

Function on column defeats index:
  WHERE UPPER(name) = 'ALICE'   → SCAN (index on name unused)
  WHERE name = 'Alice'           → SEARCH (index usable)
  WHERE strftime('%Y', order_date) = '2024'  → SCAN
  WHERE order_date >= '2024-01-01' AND order_date < '2025-01-01'  → SEARCH
""")

## Pattern 3 — Join Strategies

In [ ]:
# Simulate the three join algorithms in Python to show cost differences
# SQLite chooses automatically; understanding helps you provide the right indexes

import time

def nested_loop_join(outer, inner, outer_key, inner_key):
    """O(N x M) — one loop per outer row scans all inner rows."""
    result = []
    for o in outer:
        for i in inner:
            if o[outer_key] == i[inner_key]:
                result.append({**o, **i})
    return result

def hash_join(build_side, probe_side, build_key, probe_key):
    """O(N+M) — hash build side, probe with each probe-side row."""
    ht = {}
    for row in build_side:
        key = row[build_key]
        ht.setdefault(key, []).append(row)
    result = []
    for row in probe_side:
        for match in ht.get(row[probe_key], []):
            result.append({**match, **row})
    return result

def merge_join(left, right, left_key, right_key):
    """O(N+M) on sorted inputs — merge two sorted sequences."""
    left  = sorted(left,  key=lambda r: r[left_key])
    right = sorted(right, key=lambda r: r[right_key])
    result, j0 = [], 0
    for l in left:
        j = j0
        while j < len(right) and right[j][right_key] < l[left_key]:
            j += 1
        j0 = j
        while j < len(right) and right[j][right_key] == l[left_key]:
            result.append({**l, **right[j]})
            j += 1
    return result

# Benchmark: 500 customers x 5000 orders
N_C, N_O = 500, 5000
customers_data = [{"customer_id": i, "name": f"C{i}"} for i in range(N_C)]
orders_data = [{"order_id": i, "customer_id": i % N_C, "amount": i * 1.5} for i in range(N_O)]

for name, fn, a, b in [
    ("Nested Loop", nested_loop_join, customers_data[:50], orders_data[:500]),
    ("Hash Join",   hash_join,        customers_data,      orders_data),
    ("Merge Join",  merge_join,       customers_data,      orders_data),
]:
    t0 = time.perf_counter()
    res = fn(a, b, "customer_id", "customer_id")
    ms = (time.perf_counter() - t0) * 1000
    print(f"{name:15} rows={len(res):5}  time={ms:.2f}ms")

print("""
Selection guide:
  Nested Loop: inner table small (<1000 rows) or inner indexed
  Hash Join:   large unsorted tables, no shared sort order
  Merge Join:  both sides already sorted (e.g., on PRIMARY KEY)
  SQLite/most DBs: auto-select; provide index on join column to help
""")

## Pattern 4 — Predicate Pushdown

In [ ]:
# Filter early = fewer rows for JOIN and GROUP BY to process
# Rewrite: filter in subquery/CTE before joining, not in outer WHERE

# BAD: join all customers x all orders, then filter
slow_sql = """
    SELECT c.name, o.order_date, o.amount
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    WHERE o.status = 'completed'
      AND o.amount > 1000
      AND c.region = 'East'
"""

# GOOD: filter in subqueries/CTEs first, then join smaller sets
fast_sql = """
    WITH
    big_completed AS (
        SELECT customer_id, order_date, amount
        FROM orders
        WHERE status = 'completed' AND amount > 1000
    ),
    east_customers AS (
        SELECT customer_id, name FROM customers WHERE region = 'East'
    )
    SELECT ec.name, bc.order_date, bc.amount
    FROM east_customers ec
    JOIN big_completed bc ON ec.customer_id = bc.customer_id
"""

print("Row counts before join:")
total_orders = conn.execute("SELECT COUNT(*) FROM orders").fetchone()[0]
filtered_orders = conn.execute(
    "SELECT COUNT(*) FROM orders WHERE status='completed' AND amount>1000"
).fetchone()[0]
total_customers = conn.execute("SELECT COUNT(*) FROM customers").fetchone()[0]
east_customers = conn.execute(
    "SELECT COUNT(*) FROM customers WHERE region='East'"
).fetchone()[0]

print(f"  All orders:        {total_orders}")
print(f"  Filtered orders:   {filtered_orders}  ({100*filtered_orders//total_orders}% of all)")
print(f"  All customers:     {total_customers}")
print(f"  East customers:    {east_customers}   ({100*east_customers//total_customers}% of all)")

conn.execute("CREATE INDEX IF NOT EXISTS idx_o_status_amount ON orders(status, amount)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_c_region ON customers(region)")

print("\nExplain slow (join first, filter after):")
explain(conn, slow_sql, "Join then filter")
print("\nExplain fast (CTE pushdown):")
explain(conn, fast_sql, "CTE pushdown")

## Pattern 5 — Query Rewriting

In [ ]:
# Common rewrites that significantly improve performance

# REWRITE 1: Correlated subquery → JOIN
# Correlated subquery runs once per outer row = O(N) subquery executions
print("=== Rewrite 1: Correlated subquery → JOIN ===")

correlated_sql = """
    SELECT customer_id,
           (SELECT SUM(amount) FROM orders o2
            WHERE o2.customer_id = c.customer_id
            AND o2.status = 'completed') AS total_spend
    FROM customers c
    LIMIT 5
"""

join_sql = """
    SELECT c.customer_id, COALESCE(agg.total_spend, 0) AS total_spend
    FROM customers c
    LEFT JOIN (
        SELECT customer_id, SUM(amount) AS total_spend
        FROM orders
        WHERE status = 'completed'
        GROUP BY customer_id
    ) agg ON c.customer_id = agg.customer_id
    LIMIT 5
"""

time_query(conn, correlated_sql, "Correlated subquery (N subquery runs)")
time_query(conn, join_sql, "Rewritten as LEFT JOIN (single aggregate)")

# REWRITE 2: OR condition → UNION ALL (index-friendly)
print("\n=== Rewrite 2: OR → UNION ALL ===")

or_sql = """
    SELECT * FROM orders WHERE status = 'completed' OR status = 'cancelled'
"""
union_sql = """
    SELECT * FROM orders WHERE status = 'completed'
    UNION ALL
    SELECT * FROM orders WHERE status = 'cancelled'
"""
explain(conn, or_sql, "OR condition")
explain(conn, union_sql, "UNION ALL rewrite")

# REWRITE 3: NOT IN (subquery) → LEFT JOIN / NOT EXISTS
print("\n=== Rewrite 3: NOT IN → LEFT JOIN IS NULL ===")

not_in_sql = """
    SELECT customer_id FROM customers
    WHERE customer_id NOT IN (SELECT customer_id FROM orders)
"""
anti_join_sql = """
    SELECT c.customer_id FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    WHERE o.customer_id IS NULL
"""
time_query(conn, not_in_sql, "NOT IN subquery")
time_query(conn, anti_join_sql, "Anti-join (LEFT JOIN IS NULL)")

print("""
Why NOT IN is slow:
  If subquery returns NULL, NOT IN always returns false (3-valued logic!)
  → Use LEFT JOIN IS NULL or NOT EXISTS instead (also handles NULLs correctly)
""")

## Full Decision Map

```
SQL QUERY OPTIMIZATION CHECKLIST
──────────────────────────────────
Step 1: EXPLAIN QUERY PLAN
  SCAN = full scan (bad at scale)  →  add index or restructure
  SEARCH USING INDEX = good        →  check if covering index possible
  TEMP B-TREE = sort without index →  add index on ORDER BY column

Step 2: Index audit
  Composite index: equality cols first, range cols last
  Function on indexed col = index bypass  →  rewrite predicate
  LIKE 'prefix%' usable; LIKE '%suffix' not usable
  Low-cardinality col (status, boolean): low value alone; combine with other cols

Step 3: Filter placement
  WHERE before JOIN:   use CTE/subquery to filter each side first
  WHERE vs HAVING:     WHERE filters pre-aggregate (faster); HAVING post-aggregate
  Derived table:       SELECT * FROM (SELECT ... WHERE ...) x JOIN ...

Step 4: Rewrite patterns
  Correlated subquery  →  single aggregation + LEFT JOIN
  OR (indexed cols)    →  UNION ALL (each branch uses index)
  NOT IN               →  LEFT JOIN ... IS NULL  (handles NULLs too)
  COUNT(DISTINCT ...)  →  CTE dedupe → COUNT(*)  (often faster)
  Scalar subquery used twice → CTE defined once

Step 5: Join order & algorithm
  Small table on left in nested loop
  Index join column on larger table
  Pre-sort / pre-aggregate large side before joining
```

## Cheat Sheet

```sql
-- Read the plan
EXPLAIN QUERY PLAN SELECT ...;
-- SCAN = bad  |  SEARCH USING INDEX = good  |  COVERING INDEX = best

-- Index creation
CREATE INDEX idx_name ON table(col1, col2);           -- composite
CREATE INDEX idx_cover ON orders(customer_id, amount, status);  -- covering

-- Column order: equality first, range last
-- WHERE a=1 AND b=2 AND c>3  →  INDEX(a, b, c)

-- Avoid index bypass
-- BAD:  WHERE YEAR(order_date) = 2024
-- GOOD: WHERE order_date BETWEEN '2024-01-01' AND '2024-12-31'

-- Correlated subquery → JOIN
-- BAD:  SELECT id, (SELECT SUM(amt) FROM o WHERE o.cid=c.id) FROM c
-- GOOD: SELECT c.id, a.total FROM c LEFT JOIN
--       (SELECT cid, SUM(amt) total FROM o GROUP BY cid) a ON c.id=a.cid

-- OR → UNION ALL
-- BAD:  WHERE status='A' OR status='B'
-- GOOD: WHERE status='A' UNION ALL SELECT * ... WHERE status='B'

-- Anti-join
-- BAD:  WHERE id NOT IN (SELECT id FROM other)   -- NULL problem!
-- GOOD: LEFT JOIN other ON id=other.id WHERE other.id IS NULL

-- Predicate pushdown
WITH filtered AS (
  SELECT * FROM big_table WHERE status='active' AND region='East'
)
SELECT f.*, d.detail FROM filtered f JOIN details d ON f.id=d.id;
```

## Summary Map

```
SQL QUERY OPTIMIZATION — ONE-PAGE SUMMARY
───────────────────────────────────────────

EXPLAIN OUTPUT (SQLite)
  SCAN table          → full scan, no index
  SEARCH USING INDEX  → seeks to matching rows
  USING COVERING INDEX → all needed cols in index (no table read)
  TEMP B-TREE         → sort step (add ORDER BY-matching index)

INDEX RULES
  Equality cols → Range cols (composite left-to-right)
  Function(col) defeats index → rewrite predicate without function
  Covering = select cols + filter cols + order cols all in index

JOIN ALGORITHMS
  Nested loop: O(N×M) — fine when inner indexed or tiny
  Hash join:   O(N+M) — best for large, unsorted tables
  Merge join:  O(N+M) on sorted input — leverage existing order

REWRITES
  Correlated subquery → aggregate CTE + LEFT JOIN
  NOT IN → LEFT JOIN IS NULL  (avoids NULL-comparison bug)
  OR (multi-value) → UNION ALL
  WHERE before GROUP BY; GROUP BY before window

INTERVIEW SIGNALS
  ✓ Always EXPLAIN before guessing at the fix
  ✓ Index order: equality predicates first
  ✓ Function on column breaks index seek
  ✓ NOT IN has NULL behavior bug — use anti-join
  ✓ Correlated subquery = N round trips; rewrite as single JOIN
```